Классический бойлерплейт для работы со спарком


In [1]:
from pyspark import SparkContext, SparkConf

from pyspark.sql import SparkSession

conf = SparkConf().setAppName("SparkApp").setMaster("local")
sc = SparkContext(conf=conf)
spark = SparkSession(sc)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/19 20:11:22 WARN Utils: Your hostname, NB-Z4-PF5992Q0, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/19 20:11:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 20:11:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Регрессия

Для начала попробуем решить задачу регрессии. Возьмем хорошо известный датасет, но теперь будем работать с ним не при помощи sklearn, а используя апи спарка.
Посмотрим как выглядит наша таблица.


In [2]:
df = spark.read.csv("BostonHousing.csv", header=True, inferSchema=True)
df.show(10)

+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+
|   crim|  zn|indus|chas|  nox|   rm|  age|   dis|rad|tax|ptratio|     b|lstat|medv|
+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+
|0.00632|18.0| 2.31|   0|0.538|6.575| 65.2|  4.09|  1|296|   15.3| 396.9| 4.98|24.0|
|0.02731| 0.0| 7.07|   0|0.469|6.421| 78.9|4.9671|  2|242|   17.8| 396.9| 9.14|21.6|
|0.02729| 0.0| 7.07|   0|0.469|7.185| 61.1|4.9671|  2|242|   17.8|392.83| 4.03|34.7|
|0.03237| 0.0| 2.18|   0|0.458|6.998| 45.8|6.0622|  3|222|   18.7|394.63| 2.94|33.4|
|0.06905| 0.0| 2.18|   0|0.458|7.147| 54.2|6.0622|  3|222|   18.7| 396.9| 5.33|36.2|
|0.02985| 0.0| 2.18|   0|0.458| 6.43| 58.7|6.0622|  3|222|   18.7|394.12| 5.21|28.7|
|0.08829|12.5| 7.87|   0|0.524|6.012| 66.6|5.5605|  5|311|   15.2| 395.6|12.43|22.9|
|0.14455|12.5| 7.87|   0|0.524|6.172| 96.1|5.9505|  5|311|   15.2| 396.9|19.15|27.1|
|0.21124|12.5| 7.87|   0|0.524|5.631|100.0|6.0821|  5|311|   15.2

В спарке мы работаем с вектором фичей, соответственно для начала надо их собрать, делается это при помощи Ассемблера


In [4]:
from pyspark.ml.feature import VectorAssembler

In [5]:
assembler = VectorAssembler(
    inputCols=[
        "crim",
        "zn",
        "indus",
        "chas",
        "nox",
        "rm",
        "age",
        "dis",
        "rad",
        "tax",
        "ptratio",
        "b",
        "lstat",
    ],
    outputCol="features",
)

X = assembler.transform(df)
X.show(10)


26/03/19 20:17:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+--------------------+
|   crim|  zn|indus|chas|  nox|   rm|  age|   dis|rad|tax|ptratio|     b|lstat|medv|            features|
+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+--------------------+
|0.00632|18.0| 2.31|   0|0.538|6.575| 65.2|  4.09|  1|296|   15.3| 396.9| 4.98|24.0|[0.00632,18.0,2.3...|
|0.02731| 0.0| 7.07|   0|0.469|6.421| 78.9|4.9671|  2|242|   17.8| 396.9| 9.14|21.6|[0.02731,0.0,7.07...|
|0.02729| 0.0| 7.07|   0|0.469|7.185| 61.1|4.9671|  2|242|   17.8|392.83| 4.03|34.7|[0.02729,0.0,7.07...|
|0.03237| 0.0| 2.18|   0|0.458|6.998| 45.8|6.0622|  3|222|   18.7|394.63| 2.94|33.4|[0.03237,0.0,2.18...|
|0.06905| 0.0| 2.18|   0|0.458|7.147| 54.2|6.0622|  3|222|   18.7| 396.9| 5.33|36.2|[0.06905,0.0,2.18...|
|0.02985| 0.0| 2.18|   0|0.458| 6.43| 58.7|6.0622|  3|222|   18.7|394.12| 5.21|28.7|[0.02985,0.0,2.18...|
|0.08829|12.5| 7.87|   0|0.524|6.012| 66.6|5.5

In [6]:
X = X.select("features", "medv")
X.show(10)

+--------------------+----+
|            features|medv|
+--------------------+----+
|[0.00632,18.0,2.3...|24.0|
|[0.02731,0.0,7.07...|21.6|
|[0.02729,0.0,7.07...|34.7|
|[0.03237,0.0,2.18...|33.4|
|[0.06905,0.0,2.18...|36.2|
|[0.02985,0.0,2.18...|28.7|
|[0.08829,12.5,7.8...|22.9|
|[0.14455,12.5,7.8...|27.1|
|[0.21124,12.5,7.8...|16.5|
|[0.17004,12.5,7.8...|18.9|
+--------------------+----+
only showing top 10 rows


In [7]:
X_train, X_test = X.randomSplit([0.8, 0.2], seed=101)

Опять же, для классических моделей всё уже реализовано, можно брать и пользоваться. Соответственно работать это будет несколько непривычным образом: предикты модели мы получаем не как новый вектор, а как новый столбец в нашей таблице -- справедливая плата за возможность обучать модели на сотнях и тысячах терабайт


In [8]:
from pyspark.ml.regression import LinearRegression

In [9]:
model = LinearRegression(
    featuresCol="features", labelCol="medv", predictionCol="predicted_medv"
)
model = model.fit(X_train)

26/03/19 20:21:01 WARN Instrumentation: [a3af89f1] regParam is zero, which might cause numerical instability and overfitting.


In [10]:
predictions = model.transform(X_test)
predictions.show()

+--------------------+----+------------------+
|            features|medv|    predicted_medv|
+--------------------+----+------------------+
|[0.01311,90.0,1.2...|35.4|31.659881430440716|
|[0.01951,17.5,1.3...|33.0|  23.8615181504025|
|[0.02055,85.0,0.7...|24.7|25.091923081785886|
|[0.02543,55.0,3.7...|23.9|27.924360737323155|
|[0.02729,0.0,7.07...|34.7| 30.66453169368332|
|[0.03427,0.0,5.19...|19.5|20.006248788362672|
|[0.03548,80.0,3.6...|20.9| 21.70937064500526|
|[0.03659,25.0,4.8...|24.8|25.805677410276264|
|[0.03871,52.5,5.3...|23.2|26.845409519465477|
|[0.03932,0.0,3.41...|22.0| 27.20863438647846|
|[0.04297,52.5,5.3...|24.8|26.936734498616165|
|[0.04462,25.0,4.8...|23.9|27.042399488431396|
|[0.04544,0.0,3.24...|19.8|21.228023479480306|
|[0.0459,52.5,5.32...|22.3|27.125266942557936|
|[0.04741,0.0,11.9...|11.9|22.378089774580882|
|[0.04981,21.0,5.6...|23.4|23.718728143515396|
|[0.05083,0.0,5.19...|22.2|22.110475089995756|
|[0.0566,0.0,3.41,...|23.6|30.710921721348804|
|[0.0578,0.0,

In [11]:
from pyspark.ml.evaluation import RegressionEvaluator

In [12]:
evaluator = RegressionEvaluator(
    labelCol="medv", predictionCol="predicted_medv", metricName="rmse"
)
evaluator.evaluate(predictions)

5.577806023879184

In [13]:
evaluator = RegressionEvaluator(
    labelCol="medv", predictionCol="predicted_medv", metricName="r2"
)
evaluator.evaluate(predictions)

0.6169946126796557

In [14]:
model.coefficients

DenseVector([-0.1016, 0.051, 0.0284, 2.3159, -15.5962, 4.2673, 0.0037, -1.3962, 0.2864, -0.0124, -0.8971, 0.0095, -0.488])

### Деревья решений

Тут мы решим классификацию на примере также хорошо знакомого датасета.


In [15]:
df = spark.read.csv("iris.csv", header=True, inferSchema=True)
df.show(10)

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| setosa|
|         4.9|        3.0|         1.4|        0.2| setosa|
|         4.7|        3.2|         1.3|        0.2| setosa|
|         4.6|        3.1|         1.5|        0.2| setosa|
|         5.0|        3.6|         1.4|        0.2| setosa|
|         5.4|        3.9|         1.7|        0.4| setosa|
|         4.6|        3.4|         1.4|        0.3| setosa|
|         5.0|        3.4|         1.5|        0.2| setosa|
|         4.4|        2.9|         1.4|        0.2| setosa|
|         4.9|        3.1|         1.5|        0.1| setosa|
+------------+-----------+------------+-----------+-------+
only showing top 10 rows


Для начала, нам нужно закодировать целевой класс.


In [16]:
from pyspark.ml.feature import StringIndexer

In [17]:
indexer = StringIndexer(inputCol="species", outputCol="label")
df = indexer.fit(df).transform(df)
df.show()

+------------+-----------+------------+-----------+-------+-----+
|sepal_length|sepal_width|petal_length|petal_width|species|label|
+------------+-----------+------------+-----------+-------+-----+
|         5.1|        3.5|         1.4|        0.2| setosa|  0.0|
|         4.9|        3.0|         1.4|        0.2| setosa|  0.0|
|         4.7|        3.2|         1.3|        0.2| setosa|  0.0|
|         4.6|        3.1|         1.5|        0.2| setosa|  0.0|
|         5.0|        3.6|         1.4|        0.2| setosa|  0.0|
|         5.4|        3.9|         1.7|        0.4| setosa|  0.0|
|         4.6|        3.4|         1.4|        0.3| setosa|  0.0|
|         5.0|        3.4|         1.5|        0.2| setosa|  0.0|
|         4.4|        2.9|         1.4|        0.2| setosa|  0.0|
|         4.9|        3.1|         1.5|        0.1| setosa|  0.0|
|         5.4|        3.7|         1.5|        0.2| setosa|  0.0|
|         4.8|        3.4|         1.6|        0.2| setosa|  0.0|
|         

In [18]:
assembler = VectorAssembler(
    inputCols=[
        "sepal_length",
        "sepal_width",
        "petal_length",
        "petal_width",
    ],
    outputCol="features",
)

X = assembler.transform(df)
X.show(10)

+------------+-----------+------------+-----------+-------+-----+-----------------+
|sepal_length|sepal_width|petal_length|petal_width|species|label|         features|
+------------+-----------+------------+-----------+-------+-----+-----------------+
|         5.1|        3.5|         1.4|        0.2| setosa|  0.0|[5.1,3.5,1.4,0.2]|
|         4.9|        3.0|         1.4|        0.2| setosa|  0.0|[4.9,3.0,1.4,0.2]|
|         4.7|        3.2|         1.3|        0.2| setosa|  0.0|[4.7,3.2,1.3,0.2]|
|         4.6|        3.1|         1.5|        0.2| setosa|  0.0|[4.6,3.1,1.5,0.2]|
|         5.0|        3.6|         1.4|        0.2| setosa|  0.0|[5.0,3.6,1.4,0.2]|
|         5.4|        3.9|         1.7|        0.4| setosa|  0.0|[5.4,3.9,1.7,0.4]|
|         4.6|        3.4|         1.4|        0.3| setosa|  0.0|[4.6,3.4,1.4,0.3]|
|         5.0|        3.4|         1.5|        0.2| setosa|  0.0|[5.0,3.4,1.5,0.2]|
|         4.4|        2.9|         1.4|        0.2| setosa|  0.0|[4.4,2.9,1.

In [19]:
X = X.select("features", "label")
X.show(10)

+-----------------+-----+
|         features|label|
+-----------------+-----+
|[5.1,3.5,1.4,0.2]|  0.0|
|[4.9,3.0,1.4,0.2]|  0.0|
|[4.7,3.2,1.3,0.2]|  0.0|
|[4.6,3.1,1.5,0.2]|  0.0|
|[5.0,3.6,1.4,0.2]|  0.0|
|[5.4,3.9,1.7,0.4]|  0.0|
|[4.6,3.4,1.4,0.3]|  0.0|
|[5.0,3.4,1.5,0.2]|  0.0|
|[4.4,2.9,1.4,0.2]|  0.0|
|[4.9,3.1,1.5,0.1]|  0.0|
+-----------------+-----+
only showing top 10 rows


In [20]:
X_train, X_test = X.randomSplit([0.8, 0.2], seed=101)


In [21]:
from pyspark.ml.classification import DecisionTreeClassifier


In [26]:
model = DecisionTreeClassifier(
    labelCol="label", featuresCol="features", maxDepth=2, impurity="gini", maxBins=32
)
model = model.fit(X_train)

In [27]:
predictions = model.transform(X_test)
predictions.show(10)

+-----------------+-----+--------------+-------------+----------+
|         features|label| rawPrediction|  probability|prediction|
+-----------------+-----+--------------+-------------+----------+
|[4.5,2.3,1.3,0.3]|  0.0|[42.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|[4.8,3.4,1.9,0.2]|  0.0|[42.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|[4.9,3.0,1.4,0.2]|  0.0|[42.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|[5.0,2.0,3.5,1.0]|  1.0|[0.0,36.0,4.0]|[0.0,0.9,0.1]|       1.0|
|[5.0,2.3,3.3,1.0]|  1.0|[0.0,36.0,4.0]|[0.0,0.9,0.1]|       1.0|
|[5.1,3.5,1.4,0.3]|  0.0|[42.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|[5.2,3.4,1.4,0.2]|  0.0|[42.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|[5.4,3.4,1.5,0.4]|  0.0|[42.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|[5.4,3.9,1.7,0.4]|  0.0|[42.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|[5.5,2.3,4.0,1.3]|  1.0|[0.0,36.0,4.0]|[0.0,0.9,0.1]|       1.0|
+-----------------+-----+--------------+-------------+----------+
only showing top 10 rows


In [24]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [28]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
evaluator.evaluate(predictions)

0.9666666666666667

## Hashing Trick

Теперь давайте посмотрим что позволяет данный метод нам делать

[Датасет](https://archive.ics.uci.edu/ml/datasets/bank+marketing)

In [29]:
import pandas as pd

In [30]:
df = pd.read_csv("bank_train.csv")
labels = pd.read_csv("bank_train_target.csv", header=None)

df


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
0,26,student,single,high.school,no,no,no,telephone,jun,mon,901,1,999,0,nonexistent,1.4,94.465,-41.8,4.961,5228.1
1,46,admin.,married,university.degree,no,yes,no,cellular,aug,tue,208,2,999,0,nonexistent,1.4,93.444,-36.1,4.963,5228.1
2,49,blue-collar,married,basic.4y,unknown,yes,yes,telephone,jun,tue,131,5,999,0,nonexistent,1.4,94.465,-41.8,4.864,5228.1
3,31,technician,married,university.degree,no,no,no,cellular,jul,tue,404,1,999,0,nonexistent,-2.9,92.469,-33.6,1.044,5076.2
4,42,housemaid,married,university.degree,no,yes,no,telephone,nov,mon,85,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27590,74,retired,married,university.degree,no,yes,no,cellular,oct,fri,212,2,3,2,success,-1.1,94.601,-49.5,0.993,4963.6
27591,47,technician,divorced,professional.course,no,yes,no,cellular,jul,wed,226,1,999,0,nonexistent,1.4,93.918,-42.7,4.957,5228.1
27592,52,self-employed,married,basic.4y,unknown,yes,no,telephone,may,wed,323,7,999,0,nonexistent,1.1,93.994,-36.4,4.859,5191.0
27593,58,admin.,single,professional.course,unknown,no,no,telephone,jun,tue,203,2,999,0,nonexistent,1.4,94.465,-41.8,4.961,5228.1


In [31]:
from sklearn.preprocessing import OneHotEncoder

In [32]:
categorical_columns = df.columns[df.dtypes == 'str']
categorical_columns

Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'day_of_week', 'poutcome'],
      dtype='str')

In [33]:
onehot_encoder = OneHotEncoder(sparse_output=False)
encoded_categorical_columns = pd.DataFrame(onehot_encoder.fit_transform(df[categorical_columns]))
encoded_categorical_columns

,0,1,2,3,4,5,6,7,8,9,...,43,44,45,46,47,48,49,50,51,52
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27590,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
27591,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
27592,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
27593,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [34]:
pd.DataFrame(onehot_encoder.fit_transform(df[['job']]))

,0,1,2,3,4,5,6,7,8,9,10,11
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
27590,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
27591,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
27592,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
27593,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [45]:
hash_space_dimension = 10
print(hash('job=student') % hash_space_dimension)
print(hash('job=technician') % hash_space_dimension)


0
2


In [46]:
from pyspark.ml.feature import FeatureHasher

In [47]:
df = spark.read.csv("bank_train.csv", header=True, inferSchema=True)
df.show(10)

+---+-----------+--------+-------------------+-------+-------+----+---------+-----+-----------+--------+--------+-----+--------+-----------+------------+-----------------+-------------+---------+-----------+
|age|        job| marital|          education|default|housing|loan|  contact|month|day_of_week|duration|campaign|pdays|previous|   poutcome|emp.var.rate|   cons.price.idx|cons.conf.idx|euribor3m|nr.employed|
+---+-----------+--------+-------------------+-------+-------+----+---------+-----+-----------+--------+--------+-----+--------+-----------+------------+-----------------+-------------+---------+-----------+
| 26|    student|  single|        high.school|     no|     no|  no|telephone|  jun|        mon|     901|       1|  999|       0|nonexistent|         1.4|           94.465|        -41.8|    4.961|     5228.1|
| 46|     admin.| married|  university.degree|     no|    yes|  no| cellular|  aug|        tue|     208|       2|  999|       0|nonexistent|         1.4|           93.4

In [48]:
hasher = FeatureHasher(numFeatures=10)
hasher.setInputCols(['job'])
hasher.setOutputCol("features")
hasher.transform(df).show(10)

+---+-----------+--------+-------------------+-------+-------+----+---------+-----+-----------+--------+--------+-----+--------+-----------+------------+-----------------+-------------+---------+-----------+--------------+
|age|        job| marital|          education|default|housing|loan|  contact|month|day_of_week|duration|campaign|pdays|previous|   poutcome|emp.var.rate|   cons.price.idx|cons.conf.idx|euribor3m|nr.employed|      features|
+---+-----------+--------+-------------------+-------+-------+----+---------+-----+-----------+--------+--------+-----+--------+-----------+------------+-----------------+-------------+---------+-----------+--------------+
| 26|    student|  single|        high.school|     no|     no|  no|telephone|  jun|        mon|     901|       1|  999|       0|nonexistent|         1.4|           94.465|        -41.8|    4.961|     5228.1|(10,[4],[1.0])|
| 46|     admin.| married|  university.degree|     no|    yes|  no| cellular|  aug|        tue|     208|    

In [49]:
hasher = FeatureHasher(numFeatures=10)
hasher.setInputCols(['job', 'marital'])
hasher.setOutputCol("features")
hasher.transform(df).show(10)

+---+-----------+--------+-------------------+-------+-------+----+---------+-----+-----------+--------+--------+-----+--------+-----------+------------+-----------------+-------------+---------+-----------+--------------------+
|age|        job| marital|          education|default|housing|loan|  contact|month|day_of_week|duration|campaign|pdays|previous|   poutcome|emp.var.rate|   cons.price.idx|cons.conf.idx|euribor3m|nr.employed|            features|
+---+-----------+--------+-------------------+-------+-------+----+---------+-----+-----------+--------+--------+-----+--------+-----------+------------+-----------------+-------------+---------+-----------+--------------------+
| 26|    student|  single|        high.school|     no|     no|  no|telephone|  jun|        mon|     901|       1|  999|       0|nonexistent|         1.4|           94.465|        -41.8|    4.961|     5228.1|      (10,[4],[2.0])|
| 46|     admin.| married|  university.degree|     no|    yes|  no| cellular|  aug| 